# tensorcas — sklearn large model analysis

Measures deduplication for large GradientBoosting models:
- **Baseline**: 1 000 trees, 50 trees/step (20 checkpoints) — standard warm-start
- **Hyperparameter search**: 5 runs from the same base, each with a different `max_depth` / `learning_rate`

Key questions:
1. Does the no-op fast path still dominate at larger model sizes?
2. Do cross-run savings increase when runs share a common training set?

In [ ]:
!pip install -q git+https://github.com/Olamyy/tensorcas.git@hash-cache-no-op-path zstandard xgboost scikit-learn

In [ ]:
import copy
import pickle
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(".").resolve()))
from utils import (
    CHUNK_SIZE,
    extract_sklearn,
    measure_noop,
    measure_chunk_dedup,
    measure_crossrun,
    run_dvc_comparison,
    print_noop,
    print_chunk,
    print_crossrun,
    print_dvc_comparison,
    _fmt_bytes,
)

print("Imports OK")

## Configuration

In [ ]:
N_STEPS = 20
TREES_PER_STEP = 50   # 1 000 trees total
CHECKPOINT_DIR = Path("/tmp/tensorcas_sklearn_large")

print(f"Total trees: {N_STEPS * TREES_PER_STEP}")
print(f"Steps: {N_STEPS}, trees/step: {TREES_PER_STEP}")

## Training helpers

In [ ]:
def _make_dataset(seed: int = 0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((2000, 20)).astype(np.float32)
    y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(int)
    return X, y


def train_sequence(
    seed: int = 0,
    n_steps: int = N_STEPS,
    trees_per_step: int = TREES_PER_STEP,
    max_depth: int = 3,
    learning_rate: float = 0.1,
):
    from sklearn.ensemble import GradientBoostingClassifier
    X, y = _make_dataset(seed)
    model = GradientBoostingClassifier(
        n_estimators=trees_per_step,
        max_depth=max_depth,
        learning_rate=learning_rate,
        warm_start=True,
        random_state=seed,
    )
    models = []
    for step in range(n_steps):
        model.set_params(n_estimators=(step + 1) * trees_per_step)
        model.fit(X, y)
        models.append(copy.deepcopy(model))
    return models


print("Training helpers OK")

## Baseline: 1 000-tree warm-start run

In [ ]:
t0 = time.time()
print(f"Training {N_STEPS} checkpoints × {TREES_PER_STEP} trees (seed=0)...", end=" ", flush=True)
baseline_models = train_sequence(seed=0)
print(f"{time.time() - t0:.1f}s")

final = baseline_models[-1]
print(f"Final model: {final.n_estimators} trees")

In [ ]:
t0 = time.time()
print("Extracting tensors...", end=" ", flush=True)
baseline_seqs = [extract_sklearn(m) for m in baseline_models]
print(f"{time.time() - t0:.1f}s")

sample = baseline_seqs[-1]
n_tensors = len(sample)
total_bytes = sum(v.nbytes for v in sample.values())
print(f"Tensors per checkpoint: {n_tensors}")
print(f"Tensor bytes (final checkpoint): {_fmt_bytes(total_bytes)}")

### No-op fast path

In [ ]:
t0 = time.time()
noop_stats = measure_noop(baseline_seqs)
print_noop("sklearn large (baseline)", noop_stats)
print(f"\n  Measured in {time.time() - t0:.1f}s")

### Chunk-level reuse

In [ ]:
t0 = time.time()
chunk_stats = measure_chunk_dedup(baseline_seqs, CHUNK_SIZE)
print_chunk("sklearn large (baseline)", chunk_stats, CHUNK_SIZE)
print(f"\n  Measured in {time.time() - t0:.1f}s")

## Hyperparameter search: 5 runs from the same training set

All 5 runs use the same dataset (seed=0). Only `max_depth` and `learning_rate` vary.
Cross-run sharing measures how many chunks from previous runs can be reused.

In [ ]:
HP_GRID = [
    {"max_depth": 2, "learning_rate": 0.05},
    {"max_depth": 3, "learning_rate": 0.10},  # baseline
    {"max_depth": 4, "learning_rate": 0.10},
    {"max_depth": 3, "learning_rate": 0.20},
    {"max_depth": 5, "learning_rate": 0.05},
]

hp_models_list = []
for i, hp in enumerate(HP_GRID):
    t0 = time.time()
    print(f"  Run {i+1}/5 depth={hp['max_depth']} lr={hp['learning_rate']}...", end=" ", flush=True)
    models = train_sequence(seed=0, max_depth=hp["max_depth"], learning_rate=hp["learning_rate"])
    hp_models_list.append(models)
    print(f"{time.time() - t0:.1f}s")

In [ ]:
print("Extracting tensors for all HP runs...", end=" ", flush=True)
t0 = time.time()
hp_seqs_list = [[extract_sklearn(m) for m in models] for models in hp_models_list]
print(f"{time.time() - t0:.1f}s")

In [ ]:
# Compare each subsequent run against all prior runs combined
from utils import _hash, _chunk, _to_bytes

def _unique_hashes(seqs, chunk_size):
    s = set()
    for tensors in seqs:
        for arr in tensors.values():
            for c in _chunk(_to_bytes(arr), chunk_size):
                s.add(_hash(c))
    return s

print("\nCross-run chunk sharing (cumulative):")
print(f"  {'Run':>4} {'Config':<30} {'Unique':>8} {'Shared%':>8}")
print(f"  {'-'*4} {'-'*30} {'-'*8} {'-'*8}")

cumulative_hashes = set()
for i, (hp, seqs) in enumerate(zip(HP_GRID, hp_seqs_list)):
    run_hashes = _unique_hashes(seqs, CHUNK_SIZE)
    shared = len(run_hashes & cumulative_hashes)
    shared_pct = shared / len(run_hashes) * 100 if run_hashes else 0.0
    config = f"depth={hp['max_depth']} lr={hp['learning_rate']}"
    new_label = "(baseline)" if i == 0 else f"{shared_pct:.1f}% reused"
    print(f"  {i+1:>4} {config:<30} {len(run_hashes):>8,} {new_label:>8}")
    cumulative_hashes |= run_hashes

## DVC vs tensorcas storage comparison

Serializes the baseline run checkpoints to disk, then compares:
- **DVC-simulated**: zstd-compressed file-level dedup (what DVC would store)
- **tensorcas**: actual CAS chunk files after `tensorcasStore.save` for all steps

In [ ]:
sklearn_dir = CHECKPOINT_DIR / "sklearn"
sklearn_dir.mkdir(parents=True, exist_ok=True)

for step, model in enumerate(baseline_models, 1):
    with open(sklearn_dir / f"step_{step * TREES_PER_STEP:06d}.pkl", "wb") as f:
        pickle.dump(model, f)

files = sorted(sklearn_dir.glob("*.pkl"))
total_raw = sum(p.stat().st_size for p in files)
print(f"{len(files)} checkpoints written ({_fmt_bytes(total_raw)} total)")

In [ ]:
from tensorcas.adapters.sklearn import SklearnAdapter

print("Running DVC vs tensorcas comparison...", end=" ", flush=True)
t0 = time.time()
result = run_dvc_comparison(
    "sklearn", baseline_models, SklearnAdapter(), sklearn_dir
)
print(f"{time.time() - t0:.1f}s")

print_dvc_comparison([result])

## Summary

| Metric | Result |
|--------|--------|
| No-op rate (warm-start) | Expected ~100% for existing trees |
| Chunk reuse (changed tensors) | Depends on tree size vs chunk size |
| HP-search cross-run reuse | Increases with run count; same training set helps |
| Storage vs DVC | tensorcas wins when trees are stable across steps |

## Scaling benchmark: save latency vs tree count

Pushes sklearn warm-start to its edge by sweeping total tree counts from 1 000 to 5 000.

Measures:
- **Save latency per step** — should stay flat if tensorcas scales linearly with new tensors only
- **Tensors per checkpoint** — grows linearly (3 arrays per tree)
- **Registry overhead** — SQLite lookup count grows with total tensor history

If save latency grows super-linearly, the bottleneck is either the hash lookup table or SQLite.

In [ ]:
import statistics
import tempfile
from tensorcas.store import tensorcasStore
from tensorcas.adapters.sklearn import SklearnAdapter

# (total_trees, trees_per_step) pairs — step count varies, step size fixed at 50
SCALING_CONFIGS = [
    (500,  50),   #  10 steps
    (1000, 50),   #  20 steps  ← baseline
    (2000, 50),   #  40 steps
    (3000, 50),   #  60 steps
    (5000, 50),   # 100 steps
]

def _measure_save_latency(models, n_sample: int = 5) -> dict:
    """
    Open a single tensorcasStore and save all models in order.
    Measure wall-clock latency for the last `n_sample` steps only
    (early steps pay the first-write cost; later steps exercise the no-op path).
    Returns median, min, max in ms.
    """
    with tempfile.TemporaryDirectory() as tmp:
        adapter = SklearnAdapter()
        with tensorcasStore(root=Path(tmp), run_id="scale", adapter=adapter) as store:
            # warm-up: save all but the last n_sample steps without timing
            for step, model in enumerate(models[:-n_sample], 1):
                store.save(model, step=step)

            # timed: last n_sample steps (pure no-op path)
            latencies = []
            for i, model in enumerate(models[-n_sample:], len(models) - n_sample + 1):
                t0 = time.perf_counter()
                store.save(model, step=i)
                latencies.append((time.perf_counter() - t0) * 1000)

    return {
        "median_ms": statistics.median(latencies),
        "min_ms": min(latencies),
        "max_ms": max(latencies),
    }


print(f"{'Total trees':>12} {'Steps':>6} {'Tensors/ckpt':>14} {'Median save':>13} {'Min':>8} {'Max':>8}  {'Scaling'}")
print("=" * 80)

prev_median = None
for total_trees, trees_per_step in SCALING_CONFIGS:
    n_steps = total_trees // trees_per_step
    t0 = time.time()
    print(f"  Training {total_trees} trees ({n_steps} steps)...", end=" ", flush=True)
    models = train_sequence(seed=0, n_steps=n_steps, trees_per_step=trees_per_step)
    print(f"{time.time() - t0:.1f}s", end=" | measuring... ", flush=True)

    tensors_per_ckpt = len(extract_sklearn(models[-1]))
    lat = _measure_save_latency(models)
    print(f"done")

    flag = ""
    if prev_median is not None and lat["median_ms"] > prev_median * 2:
        flag = "  ← superlinear"
    print(
        f"  {total_trees:>12,} {n_steps:>6} {tensors_per_ckpt:>14,} "
        f"  {lat['median_ms']:>9.1f}ms {lat['min_ms']:>6.1f}ms {lat['max_ms']:>6.1f}ms{flag}"
    )
    prev_median = lat["median_ms"]

### Results (Colab, March 2026)

```
Total trees  Steps   Tensors/ckpt   Median save      Min      Max  Scaling
================================================================================
         500     10          1,500        63.5ms   57.9ms   66.0ms
       1,000     20          3,000        70.5ms   69.0ms  163.4ms
       2,000     40          6,000        83.6ms   75.6ms  103.9ms
       3,000     60          9,000        38.3ms   36.1ms   42.6ms
       5,000    100         15,000        70.2ms   59.8ms   74.2ms
```

Save latency is **flat** across all tree counts (no `← superlinear` flags). The occasional higher max values (163ms, 103ms) are OS scheduling noise, not structural growth.

**Root causes fixed** (starting from 1 425ms at 5 000 trees → 70ms, ~20×):

| Fix | Savings |
|-----|---------|
| Serial no-op partition — skip thread pool when nothing changed | ~100ms |
| Cache prev manifest in memory — skip JSON re-read each step | ~50ms |
| Hash cache fast path — skip `tensor_to_bytes` + BLAKE3 for unchanged tensors | ~200ms |
| Dtype string cache — skip `str(arr.dtype)` × 15 000 per step | ~200ms |
| Drop `refs` table — replace O(total chunks) `executemany` with manifest scan at GC time | ~1 000ms |